In [61]:
from google.colab import files
import torch
import torchaudio
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline

In [62]:
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

# CPUs generally work best with "float32" or "bfloat16".
# "float16" is primarily optimized for GPU usage.
TORCH_DTYPE = torch.float16 if torch.cuda.is_available else torch.float32

# Load audio file

Click `Browse`and load an audio file

In [ ]:
uploaded = files.upload()

In [ ]:
audio_path = list(uploaded.keys())[0]

# Whisper Large V3

In [ ]:
MODEL_ID = "openai/whisper-large-v3"

In [64]:
model = AutoModelForSpeechSeq2Seq.from_pretrained(
    MODEL_ID,
    torch_dtype=TORCH_DTYPE,
    low_cpu_mem_usage=False, # True,
    use_safetensors=True
)
model.to(DEVICE)

WhisperForConditionalGeneration(
  (model): WhisperModel(
    (encoder): WhisperEncoder(
      (conv1): Conv1d(128, 1280, kernel_size=(3,), stride=(1,), padding=(1,))
      (conv2): Conv1d(1280, 1280, kernel_size=(3,), stride=(2,), padding=(1,))
      (embed_positions): Embedding(1500, 1280)
      (layers): ModuleList(
        (0-31): 32 x WhisperEncoderLayer(
          (self_attn): WhisperSdpaAttention(
            (k_proj): Linear(in_features=1280, out_features=1280, bias=False)
            (v_proj): Linear(in_features=1280, out_features=1280, bias=True)
            (q_proj): Linear(in_features=1280, out_features=1280, bias=True)
            (out_proj): Linear(in_features=1280, out_features=1280, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=1280, out_features=5120, bias=True)
          (fc2): Linear(in_features=5120, out_features=1280, bia

In [65]:
processor = AutoProcessor.from_pretrained(MODEL_ID)

In [66]:
pipe = pipeline(
    task="automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    torch_dtype=TORCH_DTYPE,
    device=DEVICE
)

In [69]:
def load_audio(path_audio:str):
    output_sample_rate = 16_000
    waveform, sample_rate = torchaudio.load(path_audio)
    waveform = torchaudio.functional.resample(
        waveform,
        orig_freq=sample_rate,
        new_freq=output_sample_rate)
    sample = waveform.numpy()[0]
    return sample


def run_transcribe(audio_paths: list[str]) -> str:
    sample = [load_audio(audio_path) for audio_path in audio_paths]
    results = pipe(
        sample,
        batch_size=16,
        generate_kwargs={
            "language":"georgian"
        },
        # the model can predict timestamps; for sentence-level
        # timestamps, pass the "return_timestamps" argument
        return_timestamps=True
    )
    return results

In [70]:
results = run_transcribe([audio_path])

In [71]:
results[0]["text"]

' სიზარმაცეს მათხოვრობა მოსდევს იაკობ კოგებაშვილი ერთ სკოლაშის წავლებდა სანდრო. ისეთ ისარმაცი იყო, რომ ყველა სათვის თითი წაჩვენებელი შეიკმნა. სკოლაშის წავლის დროს სულ ეშმაკობდა, სკოლაშის წავლებდა სანდრო. ისეთი ზარმაცი იყო, რომ ყველასათვის თითით საჩვენებელი შეიკმნა. სკოლაშის წავლის დროს სულ ეშმაკობდა. მასწავლებელს ყურს არ უკდებდა. და ვის ამხანაკებსა ცხელ სუშლიდა. შინხომ წიგნი თვალის დასანახავა თეჯავრებოდა და სულდ თამაშობასა და ეშმაკობაში იყო კართული. ბევრი ყონის ძიება იყმარეს მასწავლებლებმა და მშოფნებმა, მაგრამ ვერაზ გახტნე, ჩგვაზე ვერ მოიყვანეს. სწავლა ვერ შიაკვარეს. საქმე იქამდე მივიდა, რომ დღეს თუხვალ სკოლიდა ნუნდა გამოერი წხად. ერთხელ სანდროს სახტამივიდა ერთი სრულია ტუცნობი მათხოვარი. სანდრო თუ მცა ზარმაცი იყო, მაგრამ გული კი კეთილიყო ქონდა. სანტრომ მათხოვარს მაშუნე პური გამოურბენინა და რამდენიმე ფული წაჯუკა. თან კიდევ გამოელაპარეგა. ბიძია, განასხვებივით მუშოავაბა არშე გიძლია? ხელი არ გაგლია და ფეხი. სიბერე შენზე ჯერ შორს არის. და ჯანითაც არა გიშავს რა? რად წანწალებდა რად დათხოულობ?

In [72]:
results[0]["chunks"][:5]

[{'timestamp': (0.0, 4.0), 'text': ' სიზარმაცეს მათხოვრობა მოსდევს'},
 {'timestamp': (4.0, 6.0), 'text': ' იაკობ კოგებაშვილი'},
 {'timestamp': (6.0, 8.0), 'text': ' ერთ სკოლაშის წავლებდა სანდრო.'},
 {'timestamp': (8.0, 10.0),
  'text': ' ისეთ ისარმაცი იყო, რომ ყველა სათვის თითი წაჩვენებელი შეიკმნა.'},
 {'timestamp': (10.0, 12.0), 'text': ' სკოლაშის წავლის დროს სულ ეშმაკობდა,'}]